# SAP Support Agent -- LangGraph + Google Gemini

An agentic support desk for SAP issues. One user complaint goes in; a triaged, grounded,
optionally escalated support response comes out.

```
START -> intake -> classify_issue -> assign_priority
                                        |
                   High ----------------+---------------- Medium / Low
                    |                                          |
                 ticket ---------------> kb_search <-----------+
                                             |
                                      system_status
                                             |
                                      draft_response
                                             |
                 needs review ---------------+--------------- no review
                    |                                          |
               human_review -> END                          final -> END
```

Concepts demonstrated: **state** with a reducer, **nodes**, **conditional edges**,
**tools**, **checkpointed memory**, and a **human-in-the-loop** approval gate.

In [ ]:
!pip install -q langgraph langchain langchain-google-genai

Paste your key from [Google AI Studio](https://aistudio.google.com/apikey).

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI Studio API key: ")

## Part 1 -- the tools

A tool is just a Python function the agent can call. These three stand in for the
systems a real SAP support desk would query: a knowledge base, a monitoring dashboard,
and a ticketing system.

In [ ]:
from __future__ import annotations

import itertools
import re
from datetime import date
from functools import lru_cache
from typing import Dict, List, Tuple

from langchain_core.tools import tool as as_tool

### The knowledge base

`SAP_KB` holds the article text and `KB_KEYWORDS` holds the words that should pull each
article up. Keeping them apart means the documentation stays readable while the matching
logic can get as clever as it needs to.

In [ ]:
# Topic -> troubleshooting steps. This is the "documentation" the agent reads.
SAP_KB: Dict[str, str] = {
    "401 unauthorized": (
        "Check destination credentials, OAuth token, client secret, role collection, "
        "and OData authentication."
    ),
    "cpi failure": (
        "Check CPI message monitoring, iFlow deployment status, certificates, "
        "and adapter configuration."
    ),
    "successfactors replication": (
        "Check employee replication job, middleware mapping, Compound Employee API, "
        "and permissions."
    ),
    "hana connection": (
        "Check HANA Cloud endpoint, IP allowlist, user privileges, and HDI container binding."
    ),
    "btp destination": (
        "In the BTP cockpit open Connectivity > Destinations, create the destination with "
        "URL, proxy type and authentication, then run 'Check Connection'."
    ),
    "s4hana api": (
        "Check the OData service in /IWFND/MAINT_SERVICE, the gateway error log "
        "/IWFND/ERROR_LOG, background jobs, and current system load."
    ),
}

# Trigger words per topic. Kept separate from SAP_KB so the article text stays
# readable -- matching is a lookup concern, not part of the documentation.
KB_KEYWORDS: Dict[str, Tuple[str, ...]] = {
    "401 unauthorized": (
        "401", "403", "unauthorized", "unauthorised", "forbidden",
        "authentication", "credential", "token", "oauth", "certificate expired",
    ),
    "cpi failure": (
        "cpi", "integration suite", "iflow", "i-flow", "cloud integration",
        "adapter", "message mapping", "middleware",
    ),
    "successfactors replication": (
        "successfactors", "employee replication", "employee data", "compound employee",
        "ec payroll", "hcm",
    ),
    "hana connection": (
        "hana", "hdi", "database connection", "cap application", "cds",
    ),
    "btp destination": (
        "destination", "btp cockpit", "connectivity", "subaccount", "role collection",
    ),
    "s4hana api": (
        "s/4hana", "s4hana", "s4 hana", "s/4 hana", "odata", "gateway",
        "order creation", "bapi", "idoc",
    ),
}

MAX_KB_HITS = 3

### The mock landscape

Every status is `Running`. Change one to `Degraded` and re-run the demo -- the drafted
response starts blaming the platform instead of the configuration, which is a quick way
to see how much the LLM is really using the facts it was handed.

In [ ]:
# Flip any value to "Degraded" or "Down" to watch the agent's draft response
# start blaming the platform instead of the configuration.
MOCK_STATUS: Dict[str, str] = {
    "CPI": "Running",
    "SuccessFactors": "Running",
    "S4HANA": "Running",
    "HANA": "Running",
    "BTP": "Running",
}

# Every spelling a user might type, mapped onto a MOCK_STATUS key.
SYSTEM_ALIASES: Dict[str, str] = {
    "cpi": "CPI",
    "integration suite": "CPI",
    "cloud integration": "CPI",
    "iflow": "CPI",
    "successfactors": "SuccessFactors",
    "employee central": "SuccessFactors",
    "s4hana": "S4HANA",
    "s/4hana": "S4HANA",
    "s4 hana": "S4HANA",
    "s/4 hana": "S4HANA",
    "ecc": "S4HANA",
    "hana": "HANA",
    "hana cloud": "HANA",
    "hdi": "HANA",
    "btp": "BTP",
    "business technology platform": "BTP",
    "cockpit": "BTP",
}

_TICKET_SEQ = itertools.count(1)


@lru_cache(maxsize=None)
def _pattern(phrase: str) -> re.Pattern:
    """Whole-word matcher for one keyword.

    Plain ``in`` matching is what makes "S/4HANA" look like a HANA Cloud issue
    and "Salesforce" look like SuccessFactors. Word boundaries stop both: in
    "s/4hana" the "hana" sits directly after the word character "4", so no
    boundary exists there, while "HANA Cloud" still matches.
    """
    return re.compile(rf"\b{re.escape(phrase)}\b")


def mentions(text: str, phrase: str) -> bool:
    """True when ``phrase`` appears in ``text`` as a whole word."""
    return _pattern(phrase.lower()).search(text.lower()) is not None

### Tool 1 -- `search_kb`

Scores every article by how many of its trigger words appear, then returns the best
three. Matching is whole-word, not substring -- with plain `in` checks,
*"Production S/4HANA order creation API is down"* also matched the **hana connection**
article, because `"hana" in "s/4hana"` is `True`.

In [ ]:
def search_kb(query: str) -> str:
    """Search the SAP support knowledge base and return troubleshooting steps.

    Scores every article by how many of its trigger words appear in the query and
    returns the best matches, so a single sentence mentioning both CPI and a 401
    pulls back both articles.
    """
    scored = [
        (sum(mentions(query, word) for word in words), rank, topic)
        for rank, (topic, words) in enumerate(KB_KEYWORDS.items())
    ]
    # Best score first; ties keep SAP_KB order rather than falling back to
    # alphabetical, so the same query always returns the same article order.
    hits = sorted((s for s in scored if s[0] > 0), key=lambda s: (-s[0], s[1]))[:MAX_KB_HITS]

    if not hits:
        return "No knowledge base article matched. Handle as a new issue and document the fix."

    return "\n".join(f"[{topic}] {SAP_KB[topic]}" for _, _, topic in hits)

### Tool 2 -- `check_system_status`

Users write `S/4HANA`, `s4 hana` and `ECC` for the same box, so aliases are resolved
before lookup. `detect_systems` reads the issue text and returns every system mentioned,
longest alias first so `s/4hana` is consumed before bare `hana` can match it.

In [ ]:
def check_system_status(system_name: str) -> str:
    """Return the mock availability of one SAP system, e.g. ``CPI = Running``."""
    key = SYSTEM_ALIASES.get(system_name.strip().lower())
    if key is None:
        # Longest alias first, so "s/4hana" is settled before bare "hana".
        key = next(
            (
                SYSTEM_ALIASES[alias]
                for alias in sorted(SYSTEM_ALIASES, key=len, reverse=True)
                if mentions(system_name, alias)
            ),
            None,
        )
    if key is None:
        return f"{system_name} = Unknown system"
    return f"{key} = {MOCK_STATUS[key]}"


def detect_systems(text: str) -> List[str]:
    """Return the MOCK_STATUS keys mentioned in free text, longest alias first.

    Longest-first matching stops "hana" inside "s/4hana" from registering HANA
    Cloud when the user only ever mentioned S/4HANA.
    """
    lowered = text.lower()
    found: List[str] = []
    for alias in sorted(SYSTEM_ALIASES, key=len, reverse=True):
        matched, count = _pattern(alias).subn(" ", lowered)
        if count:
            lowered = matched  # consumed, so a shorter alias cannot re-match it
            system = SYSTEM_ALIASES[alias]
            if system not in found:
                found.append(system)
    return found

### Tool 3 -- `create_support_ticket`

Returns an ID like `INC-20260721-P1-001`. The print statement is the giveaway that the
High-priority branch actually ran.

In [ ]:
def create_support_ticket(summary: str, priority: str) -> str:
    """Create a mock support ticket and return its ticket ID."""
    band = {"high": "P1", "medium": "P2", "low": "P3"}.get(priority.strip().lower(), "P3")
    ticket_id = f"INC-{date.today():%Y%m%d}-{band}-{next(_TICKET_SEQ):03d}"
    print(f"[ticket] {ticket_id} raised ({priority}): {summary[:70].strip()}...")
    return ticket_id

### Tools as LangChain objects

The graph calls these functions directly, so it does not need them wrapped. This export
exists for the advanced challenge, where Gemini decides which tool to call and `ToolNode`
executes it.

In [ ]:
SAP_TOOLS = [as_tool(search_kb), as_tool(check_system_status), as_tool(create_support_ticket)]

## Part 2 -- the agent

Categories are declared twice: once as a list for the prompt, once as a `Literal` for the
structured output schema. The `Literal` is what stops Gemini returning
*"SAP Integration Suite (CPI)"* and silently breaking a router that compares strings.

In [ ]:
from __future__ import annotations

import operator
import os
import sys
from typing import Annotated, Callable, List, Literal, Optional, TypedDict


from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field





# Overridable so the project keeps working as new Gemini versions ship.
CHAT_MODEL = os.getenv("GEMINI_CHAT_MODEL", "gemini-2.5-flash")

CATEGORIES = [
    "SAP BTP",
    "SAP Integration Suite / CPI",
    "SAP SuccessFactors",
    "SAP S/4HANA",
    "SAP HANA Cloud",
    "SAP Build Process Automation",
    "General SAP",
]

Category = Literal[
    "SAP BTP",
    "SAP Integration Suite / CPI",
    "SAP SuccessFactors",
    "SAP S/4HANA",
    "SAP HANA Cloud",
    "SAP Build Process Automation",
    "General SAP",
]
Priority = Literal["High", "Medium", "Low"]

### State

Every node receives this dict and returns only the keys it changed; LangGraph merges the
result. Plain keys are overwritten -- but `history` carries the `operator.add` reducer, so
its lists are concatenated instead. That one annotation is the whole difference between an
agent that forgets and an agent that accumulates.

In [ ]:
class SupportAgentState(TypedDict, total=False):
    """Shared memory every node reads from and writes to.

    A node returns a partial dict; LangGraph merges it into the state. Plain keys
    are overwritten, but ``history`` carries the ``operator.add`` reducer, so its
    lists are concatenated instead -- that is what lets one thread build up a
    record of every issue it has seen.
    """

    user_issue: str
    category: Optional[str]
    priority: Optional[str]
    kb_result: Optional[str]
    system_status: Optional[str]
    draft_response: Optional[str]
    final_response: Optional[str]
    ticket_id: Optional[str]
    needs_human_review: bool
    history: Annotated[List[str], operator.add]

### Gemini, and the two decisions that drive the graph

`with_structured_output` binds a Pydantic schema to the model, so classification and
prioritisation come back as validated objects rather than prose to parse.

The three `_*_with_llm` helpers are separated from the nodes on purpose: it lets the
routing be tested with the LLM stubbed out, which is what `test_routing.py` does.

In [ ]:
_llm = None


def get_llm():
    """Build the Gemini client once and reuse it across nodes."""
    global _llm
    if _llm is None:
        if not os.getenv("GOOGLE_API_KEY"):
            raise RuntimeError(
                "GOOGLE_API_KEY is not set. Copy .env.example to .env and paste your "
                "key from https://aistudio.google.com/apikey"
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        _llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0)
    return _llm


class IssueClassification(BaseModel):
    """Structured output for the classification node."""

    category: Category = Field(description="The SAP area that owns this issue.")


class PriorityDecision(BaseModel):
    """Structured output for the priority node."""

    priority: Priority = Field(description="High, Medium or Low.")
    reason: str = Field(description="One sentence justifying the priority.")


CLASSIFY_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an SAP support triage assistant. Assign the incoming issue to "
            "exactly one category from this list:\n{categories}\n\n"
            "Pick the area that owns the fix, not every product mentioned. An "
            "integration flow failing between two systems belongs to "
            "'SAP Integration Suite / CPI'. Use 'General SAP' only when no other "
            "category fits.",
        ),
        ("human", "Issue:\n{issue}"),
    ]
)

PRIORITY_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an SAP support triage assistant deciding incident priority.\n\n"
            "High   -- production is down, a business process is stopped, data "
            "replication has stopped, payments/orders are blocked, or many users "
            "are affected.\n"
            "Medium -- something is broken but contained: one user, one "
            "non-production system, or a workaround exists.\n"
            "Low    -- a how-to, configuration or general question. Nothing is broken.\n\n"
            "Examples:\n"
            "'Payroll posting to S/4HANA has stopped for all employees.' -> High\n"
            "'One user cannot open a Fiori tile, others can.' -> Medium\n"
            "'Where do I find the CPI message monitoring screen?' -> Low",
        ),
        ("human", "Issue:\n{issue}"),
    ]
)

DRAFT_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a senior SAP support engineer. Write the response the support "
            "desk will send back, grounded in the knowledge base extract and the "
            "system status provided. Do not invent log entries or error codes that "
            "were not supplied.\n\n"
            "Reply in exactly this format, with no extra commentary:\n\n"
            "Issue Category: <category>\n"
            "Priority: <priority>\n"
            "Likely Cause: <one or two sentences>\n"
            "Suggested Resolution:\n"
            "1. <step>\n"
            "2. <step>\n"
            "3. <step>\n"
            "4. <step>\n"
            "5. <step>\n"
            "Escalation Required: <Yes or No>",
        ),
        (
            "human",
            "User issue:\n{issue}\n\n"
            "Category: {category}\n"
            "Priority: {priority}\n"
            "Escalation required: {escalation}\n"
            "Ticket: {ticket}\n\n"
            "Knowledge base extract:\n{kb_result}\n\n"
            "System status:\n{system_status}",
        ),
    ]
)


def _classify_with_llm(issue: str) -> str:
    """Ask Gemini for the category. Split out so tests can stub the LLM."""
    chain = CLASSIFY_PROMPT | get_llm().with_structured_output(IssueClassification)
    return chain.invoke({"issue": issue, "categories": "\n".join(CATEGORIES)}).category


def _prioritise_with_llm(issue: str) -> str:
    """Ask Gemini for the priority. Split out so tests can stub the LLM."""
    chain = PRIORITY_PROMPT | get_llm().with_structured_output(PriorityDecision)
    return chain.invoke({"issue": issue}).priority


def _draft_with_llm(state: SupportAgentState) -> str:
    """Ask Gemini for the customer-facing response."""
    chain = DRAFT_PROMPT | get_llm()
    return chain.invoke(
        {
            "issue": state["user_issue"],
            "category": state.get("category"),
            "priority": state.get("priority"),
            "escalation": "Yes" if state.get("needs_human_review") else "No",
            "ticket": state.get("ticket_id") or "none raised",
            "kb_result": state.get("kb_result"),
            "system_status": state.get("system_status"),
        }
    ).content.strip()

### Nodes

One node, one job. Note that no node calls another -- they only read and write state,
which is why adding `system_status` to the flow required no change to any other node.

In [ ]:
def intake_node(state: SupportAgentState) -> dict:
    """Node 1 -- clean the raw issue and record it in the thread history."""
    issue = " ".join(state["user_issue"].split())
    return {"user_issue": issue, "history": [issue], "needs_human_review": False}


def classify_node(state: SupportAgentState) -> dict:
    """Node 2 -- Gemini assigns one of the seven SAP categories."""
    return {"category": _classify_with_llm(state["user_issue"])}


def priority_node(state: SupportAgentState) -> dict:
    """Node 3 -- Gemini assigns priority; High also flags human review."""
    priority = _prioritise_with_llm(state["user_issue"])
    return {"priority": priority, "needs_human_review": priority == "High"}


def kb_search_node(state: SupportAgentState) -> dict:
    """Node 4 -- tool call into the knowledge base."""
    return {"kb_result": search_kb(state["user_issue"])}


def system_status_node(state: SupportAgentState) -> dict:
    """Node 5 -- tool call for every system named in the issue."""
    systems = detect_systems(state["user_issue"]) or ["BTP"]
    return {"system_status": "; ".join(check_system_status(s) for s in systems)}


def draft_response_node(state: SupportAgentState) -> dict:
    """Node 6 -- Gemini writes the technical response from the gathered facts."""
    return {"draft_response": _draft_with_llm(state)}


def human_review_node(state: SupportAgentState) -> dict:
    """Node 7 -- a human approves or rejects the draft before it goes out."""
    print("\nHuman review required.")
    print(state["draft_response"])
    if _approver(state):
        return {"final_response": state["draft_response"]}
    return {"final_response": "Response rejected. Please revise with more details."}


def ticket_node(state: SupportAgentState) -> dict:
    """Node 8 -- raise a ticket. Only reached on the High-priority branch."""
    ticket_id = create_support_ticket(
        summary=state["user_issue"],
        priority=state["priority"],
    )
    return {"ticket_id": ticket_id}


def final_node(state: SupportAgentState) -> dict:
    """Node 9 -- Medium/Low responses ship without a human in the loop."""
    return {"final_response": state["draft_response"]}

### Human-in-the-loop

`input()` is called only when a console is attached. In Colab there is none, so the review
auto-approves rather than hanging the cell. Use `set_approver` to plug in your own
decision -- a real prompt, an always-reject, or a Slack round-trip.

In [ ]:
def _prompt_approver(state: SupportAgentState) -> bool:
    """Ask at the console, but auto-approve when there is nobody to ask.

    Two separate guards, because they catch different failures. The isatty check
    covers notebooks and schedulers with no console at all. The EOFError catch
    covers the sneakier case of a stdin that reports itself as a terminal but has
    nothing to read -- some CI runners and sandboxes do exactly that, and without
    the catch the whole graph dies at the review step.
    """
    if sys.stdin is None or not sys.stdin.isatty():
        print("Approve response? yes/no: yes  (no console attached, auto-approved)")
        return True
    try:
        return input("Approve response? yes/no: ").strip().lower() in {"y", "yes"}
    except EOFError:
        print("yes  (nothing on stdin, auto-approved)")
        return True


_approver: Callable[[SupportAgentState], bool] = _prompt_approver


def set_approver(fn: Callable[[SupportAgentState], bool]) -> None:
    """Swap the approval step -- used by run_demo.py and the tests.

    Kept as a module setting rather than a graph config value because a callable
    cannot be serialised into a checkpoint.
    """
    global _approver
    _approver = fn

### Conditional edges

A router is a function that reads state and returns the *name* of the next node. Keeping
it separate from the node that made the decision is what makes the routing independently
testable.

In [ ]:
def route_after_priority(state: SupportAgentState) -> str:
    """High priority earns a ticket first; everything else goes straight to the KB."""
    return "ticket" if state["priority"] == "High" else "kb_search"


def route_after_draft(state: SupportAgentState) -> str:
    """Escalated drafts need a human signature before they are final."""
    return "human_review" if state.get("needs_human_review") else "final"

### The graph

Nine nodes, two conditional edges. `build_agent(memory=True)` attaches an `InMemorySaver`,
after which invoking with the same `thread_id` resumes the stored state instead of
starting fresh.

In [ ]:
def build_agent(memory: bool = False):
    """Wire the nodes and edges together and compile the graph.

    ``memory=True`` attaches an InMemorySaver, so invoking with the same
    ``thread_id`` resumes the stored state instead of starting from scratch.
    """
    workflow = StateGraph(SupportAgentState)

    workflow.add_node("intake", intake_node)
    workflow.add_node("classify_issue", classify_node)
    workflow.add_node("assign_priority", priority_node)
    workflow.add_node("ticket", ticket_node)
    workflow.add_node("kb_search", kb_search_node)
    workflow.add_node("system_status", system_status_node)
    workflow.add_node("draft_response", draft_response_node)
    workflow.add_node("human_review", human_review_node)
    workflow.add_node("final", final_node)

    workflow.add_edge(START, "intake")
    workflow.add_edge("intake", "classify_issue")
    workflow.add_edge("classify_issue", "assign_priority")
    workflow.add_conditional_edges(
        "assign_priority",
        route_after_priority,
        {"ticket": "ticket", "kb_search": "kb_search"},
    )
    workflow.add_edge("ticket", "kb_search")
    workflow.add_edge("kb_search", "system_status")
    workflow.add_edge("system_status", "draft_response")
    workflow.add_conditional_edges(
        "draft_response",
        route_after_draft,
        {"human_review": "human_review", "final": "final"},
    )
    workflow.add_edge("human_review", END)
    workflow.add_edge("final", END)

    return workflow.compile(checkpointer=InMemorySaver() if memory else None)


def run_agent(issue: str, app=None, thread_id: Optional[str] = None) -> SupportAgentState:
    """Run one issue through the graph and return the final state.

    Pass ``thread_id`` together with an ``app`` built by ``build_agent(memory=True)``
    to keep several issues on one remembered thread.
    """
    app = app or build_agent(memory=thread_id is not None)
    config = {"configurable": {"thread_id": thread_id}} if thread_id else {}
    return app.invoke({"user_issue": issue}, config=config)

### Reporting

In [ ]:
def format_result(state: SupportAgentState) -> str:
    """Render the run as the scorecard from the problem statement."""
    lines = [
        f"Issue            : {state['user_issue']}",
        f"Category         : {state.get('category')}",
        f"Priority         : {state.get('priority')}",
        f"Ticket Created   : {'Yes (' + state['ticket_id'] + ')' if state.get('ticket_id') else 'No'}",
        f"Human Review     : {'Yes' if state.get('needs_human_review') else 'No'}",
        f"System Status    : {state.get('system_status')}",
        "",
        state.get("final_response") or "(no response produced)",
    ]
    return "\n".join(lines)


def print_graph() -> None:
    """Print the compiled graph as Mermaid -- paste it into any Markdown viewer."""
    print(build_agent().get_graph().draw_mermaid())

### The compiled graph

Rendered from the graph object itself, so it cannot go stale.

In [ ]:
try:
    from IPython.display import Image, display

    display(Image(build_agent().get_graph().draw_mermaid_png()))
except Exception:
    print(build_agent().get_graph().draw_mermaid())

## Part 3 -- the four test cases

Category and priority come from Gemini, so treat the expected values as the target rather than a guarantee. A disagreement is a prompt to sharpen the rules in `PRIORITY_PROMPT`.

In [ ]:
app = build_agent()

TEST_CASES = [
    ("SAP CPI integration from SuccessFactors to S/4HANA is failing with 401 "
     "unauthorized error. Employee replication has stopped.",
     "SAP Integration Suite / CPI | High | ticket | review"),
    ("How can I create a destination in SAP BTP cockpit?",
     "SAP BTP | Low | no ticket | no review"),
    ("SAP HANA Cloud connection is failing from CAP application.",
     "SAP HANA Cloud | Medium | no ticket | no review"),
    ("Production S/4HANA order creation API is down and sales users cannot create orders.",
     "SAP S/4HANA | High | ticket | review"),
]

for i, (issue, expected) in enumerate(TEST_CASES, start=1):
    print("=" * 78)
    print(f"Test Case {i}   expected -> {expected}")
    print("=" * 78)
    print(format_result(run_agent(issue, app=app)))
    print()

## Part 4 -- memory

Two issues, one `thread_id`. The checkpointer restores the stored state before the second run, and the `operator.add` reducer on `history` keeps both.

In [ ]:
remembering = build_agent(memory=True)

run_agent(TEST_CASES[0][0], app=remembering, thread_id="ticket-4711")
state = run_agent(
    "The same CPI iFlow is still failing after we rotated the client secret.",
    app=remembering,
    thread_id="ticket-4711",
)

print("Issues seen on thread ticket-4711:")
for n, past in enumerate(state["history"], start=1):
    print(f"  {n}. {past}")

## Where to take it next

- Move `ticket` after `draft_response` so the incident includes the suggested fix.
- Route a rejected review back to `draft_response` with the reviewer's comment, making it
  a real revision loop.
- Swap `human_review` for LangGraph's `interrupt()` so the graph pauses and resumes from
  the checkpoint instead of blocking on `input()`.
- Let Gemini pick the tools with `ToolNode` + `tools_condition` instead of fixed nodes.
- Split the workflow into classifier / resolver / escalation / writer agents.